# 🚀 ViSceT5 — PreSTU SplitOCR Pre-Training trên Kaggle

Notebook này hướng dẫn quy trình tiền huấn luyện (**PreSTU SplitOCR**) mô hình **ViSceT5** trên 2 bộ dữ liệu Scene-Text tiếng Việt: **VinText** và **EVJVQA**.

### 📌 Đặc điểm của phương pháp PreSTU (Google Research, 2023):
1. **Chỉ nhận Image Pixels + Text Prompt:** Mô hình encoder chỉ nhận ảnh điểm ảnh từ CLIP-ViT và chuỗi văn bản tiền tố (prefix text).
2. **Cơ chế SplitOCR:** Sắp xếp OCR theo không gian ($	ext{Top-Left} \rightarrow \text{Bottom-Right}$), chọn điểm cắt ngẫu nhiên $m \in [0, N-1]$.
   * $m = 0$: Pure OCR (Prompt: `"Generate ocr_text in vi:"` $\rightarrow$ Target: Toàn bộ $N$ từ OCR).
   * $m > 0$: Split Continuation (Prompt: `"Generate ocr_text in vi: <OCR_1>...<OCR_m>"` $\rightarrow$ Target: `"<OCR_{m+1}>...<OCR_N>"`).
3. **Tối ưu trực tiếp qua Decoder:** Sử dụng hàm Cross-Entropy Loss trực tiếp trên chuỗi mục tiêu do ViT5 Decoder sinh ra.

## 1. Clone Codebase & Cài Đặt Môi Trường

In [ ]:
# Clone repo và checkout đúng nhánh tiền huấn luyện exp/pretrain-gen-all
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git fetch origin
!git checkout exp/pretrain-gen-all
!git pull origin exp/pretrain-gen-all
!git log --oneline -1

In [ ]:
# Cài đặt các thư viện phụ thuộc
%%capture
!pip install -q transformers accelerate gdown sentencepiece safetensors ftfy regex torchvision evaluate

## 2. Chuẩn Bị & Khớp Nối Dữ Liệu (VinText + EVJVQA)
Hệ thống sẽ tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và phân chia train/validation.

In [ ]:
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 3. Khởi Tạo Trọng Số Mô Hình (ViT5 Base)

In [ ]:
from scripts import init_model
init_model.main()

## 4. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

* **Epochs:** 10
* **Batch size:** 4 (per device) x 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** 1e-4 với warmup
* **Thư mục lưu:** `/kaggle/working/pretrain_output`

In [ ]:
!python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --num_train_epochs 10 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.0001 \
    --save_total_limit 1 \
    --output_dir /kaggle/working/pretrain_output \
    --logging_dir /kaggle/working/pretrain_output/logs

## 5. Nén Checkpoint Để Tải Về / Upload Lên Hugging Face Hub

In [ ]:
# Nén best model thành file ZIP để dễ dàng tải về từ giao diện Kaggle
!zip -r /kaggle/working/ViSceT5_PreSTU_Best.zip /kaggle/working/pretrain_output/best_model
print("✅ Đã nén thành công checkpoint tại /kaggle/working/ViSceT5_PreSTU_Best.zip")

In [ ]:
# (Tùy chọn) Đăng tải trực tiếp lên HuggingFace Hub nếu có Token
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="/kaggle/working/pretrain_output/best_model",
#     repo_id="your-username/ViSceT5-PreSTU-Pretrained",
#     repo_type="model",
#     token="your_hf_token_here"
# )